# CCR follow-up: replay, time steps and the 29-cell lesion

Extract CCR_Followup.zip into /user/cbaker4/connectome. Keep the original results/expanded_sensitivity folder and .ccr-venv. Use Run All once.

Request UB-HPC, smuldoon, general-compute, matching general-compute QOS, 24 cores, 192000 MB RAM, 6 hours, no GPU. Leave extra modules and node features blank for the supported Jupyter app. The saved ccr_python.json provides the Python 3.11 module environment for subprocesses; the notebook kernel may remain Python 3.9. No new packages are required.

There are 101 trials: 25 exact replays, 46 trials at smaller time steps, and 30 active-29 lesions. Up to 16 workers run at once. A replay mismatch stops later stages. Smaller-step results are collected without an invented pass threshold. Completion means execution and checks finished, not that the model has converged.


In [ ]:
from pathlib import Path
import os, json, sys, subprocess, time, hashlib
ROOT = Path.cwd().resolve()
STUDY = ROOT / 'results/expanded_sensitivity'
OUT = ROOT / 'results/followup_20260927'
PYTHON = ROOT / '.ccr-venv/bin/python'
SCRIPT = ROOT / 'scripts/pcdr_followup.py'
RUN_HOURS = 5
LOG = ROOT / (OUT.name + '_controller.log')
if not (ROOT / 'transfer_manifest.json').is_file() or not (STUDY / 'jobs.json').is_file() or not PYTHON.is_file():
    raise RuntimeError('Open this notebook inside the existing /user/cbaker4/connectome directory.')
if hashlib.sha256((ROOT / 'followup_plan.json').read_bytes()).hexdigest() != '1d074df882a5bb71a7c3dd6bc52bf2928335b146d1ada956a3ac0680fd1c2613':
    raise RuntimeError('Follow-up plan differs from the reviewed package.')
plan = json.loads((ROOT / 'followup_plan.json').read_text())
for name, expected in plan['sources'].items():
    if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Changed follow-up source: ' + name)
config = json.loads((ROOT / 'ccr_python.json').read_text())
RUN_ENV = os.environ.copy()
for key, value in config.get('env', {}).items():
    if value is None:
        RUN_ENV.pop(key, None)
    else:
        RUN_ENV[key] = value
for key in ['PYTHONHOME', 'PYTHONPATH']:
    RUN_ENV.pop(key, None)
RUN_ENV.update(PYTHONNOUSERSITE='1', MPLBACKEND='Agg', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
subprocess.run([str(PYTHON), '-c', 'import sys,numpy,pandas,pyarrow,brian2; assert sys.version_info[:2]==(3,11); print(sys.version)'], env=RUN_ENV, check=True)
print('Original study:', STUDY)
print('New output:', OUT)
print('Trials:', len(plan['jobs']))


The next cell starts a separate process with a file log, then waits for it. Closing the browser does not stop that process while the allocation stays alive. Do not click Run All a second time while it is running. Logs and progress stay on disk. If the allocation ends, the process stops; a new allocation is not automatic.

Existing output folders are never overwritten. If the run fails, keep the error and results ZIP for review. Do not change the Python versions or delete old outputs to make it proceed. A completed run can be reopened and this notebook rerun to show its download location.


In [ ]:
if OUT.exists():
    record = json.loads((OUT / 'progress.json').read_text())
    if record.get('status') != 'complete':
        raise RuntimeError('An existing run is unfinished or failed. Read ' + str(LOG) + ' and ' + str(OUT / 'progress.json') + '; do not launch another copy.')
else:
    command = [str(PYTHON), '-u', str(SCRIPT), 'run', '--study', str(STUDY), '--out', str(OUT), '--hours', str(RUN_HOURS)]
    with LOG.open('x', encoding='utf-8') as stream:
        process = subprocess.Popen(command, cwd=ROOT, env=RUN_ENV, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    print('Controller PID:', process.pid, 'Log:', LOG, flush=True)
    position = 0
    while True:
        with LOG.open(encoding='utf-8', errors='replace') as stream:
            stream.seek(position)
            text = stream.read()
            position = stream.tell()
        if text:
            print(text, end='', flush=True)
        result = process.poll()
        if result is not None:
            with LOG.open(encoding='utf-8', errors='replace') as stream:
                stream.seek(position)
                print(stream.read(), end='', flush=True)
            break
        time.sleep(5)
    if result:
        print('If present, download:', OUT / 'CCR_followup_results.zip')
        raise RuntimeError('Run stopped. Keep the log and any result ZIP; the original data are unchanged.')
record = json.loads((OUT / 'progress.json').read_text())
if record['status'] != 'complete':
    raise RuntimeError('Collection is not complete.')
print('Finished:', record['completed'], 'trials')
print('Download:', OUT / 'CCR_followup_results.zip')


To check from a terminal:

```bash
tail -n 20 /user/cbaker4/connectome/followup_20260927_controller.log
cat /user/cbaker4/connectome/results/followup_20260927/progress.json
```

Download results/followup_20260927/CCR_followup_results.zip and the controller log when finished. The ZIP contains sparse rates, population traces, summaries, settings, source files and worker logs. Full spikes and delivered-event tables remain in the new trial folders on CCR. Missing rows in each sparse rate table mean zero, using neurons.csv as the full universe. Keep those trial folders until the results have been reviewed. Analysis of this package does not fix the earlier comparison-set imbalance.
